# Vectorless RAG
### Reasoning-based RAG with No Vector DB, No Chunking

---

## 🧠 What You'll Learn

| # | Topic |
|---|-------|
| 1 | Why Vector RAG fails on professional documents |
| 2 | How PageIndex builds a tree index from a PDF |
| 3 | LLM Tree Search — reasoning over structure |
| 4 | Full end-to-end Vectorless RAG pipeline |
| 5 | Expert-guided retrieval (domain knowledge injection) |
| 6 | Chat API — zero LLM setup |
| 7 | Self-hosted open-source option |

---

## 🔑 Key Concept

> **Traditional RAG** → chunk → embed → cosine similarity → retrieve  
> **PageIndex RAG** → build tree → LLM reasons over tree → retrieve exact sections

**The problem with vector RAG:**  
`Similarity ≠ Relevance`  
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.

In [16]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
BAZZAAR_API_KEY = os.getenv("BAZARRLINK")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Bazaar key loaded:   ", "✅" if BAZZAAR_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Bazaar key loaded:    ✅


In [25]:
from pageindex import PageIndexClient
from openai import OpenAI
pi_client=PageIndexClient(api_key=PAGEINDEX_API_KEY)
ai_client=OpenAI(base_url="https://api.bazaarlink.ai/v1", api_key=BAZZAAR_API_KEY,
                 )


# Section 2: Upload & Index a PDF


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [18]:
PDF_PATH = r"D:\ProjectsGit\ai-ml\RAG\BasicRAG\data\pdf\attention_is_all_you_need.pdf"

result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

✅ Uploaded!
📋 Document ID: pi-cmvh5gosd007i0bnz5mzpff65
   (Save this ID — you'll use it throughout the notebook)


C:\Users\HP\AppData\Local\Temp\ipykernel_31308\1662534302.py:3: UserWarning: Document "attention_is_all_you_need.pdf" was stored as "attention_is_all_you_need_1.pdf".
  result = pi_client.submit_document(PDF_PATH)


In [19]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [8]:
#fetch full tree 
tree_result=pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree=tree_result.get('result',[])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 15,
  "summary": "This section, titled \"Attention Is All You Need,\" presents the foundational research paper introducing the Transformer, a novel neural network architecture for sequence modeling based entirely on attention mechanisms rather than recurrence or convolutions. It covers the authors and affiliations, an abstract and introduction outlining the motivation to eliminate sequential computation for better parallelization, necessary background, and the detailed model architecture including encoder-decoder structures, self-attention, and positional encodings. Furthermore, it justifies the use of self-attention through complexity and path-length comparisons, outlines the training regime, presents experimental results and ablation studies on machine translation and constituency parsing, offers conclusions and future directions, lists r

In [9]:

# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Attention Is All You Need  (p.?)
  └─ [0001] Attention Is All You Need (intro)  (p.?)
  └─ [0002] Abstract  (p.?)
  └─ [0003] 1 Introduction  (p.?)
  └─ [0004] 2 Background  (p.?)
  └─ [0005] 3 Model Architecture  (p.?)
    └─ [0006] 3 Model Architecture (intro)  (p.?)
    └─ [0007] 3.1 Encoder and Decoder Stacks  (p.?)
    └─ [0008] 3.2 Attention  (p.?)
      └─ [0009] 3.2 Attention (intro)  (p.?)
      └─ [0010] 3.2.1 Scaled Dot-Product Attention  (p.?)
      └─ [0011] 3.2.2 Multi-Head Attention  (p.?)
      └─ [0012] 3.2.3 Applications of Attention in our Model  (p.?)
    └─ [0013] 3.3 Position-wise Feed-Forward Networks  (p.?)
    └─ [0014] 3.4 Embeddings and Softmax  (p.?)
    └─ [0015] 3.5 Positional Encoding  (p.?)
  └─ [0016] 4 Why Self-Attention  (p.?)
  └─ [0017] 5 Training  (p.?)
  └─ [0018] 6 Results  (p.?)
    └─ [0019] 6.1 Machine Translation  (p.?)
    └─ [0020] 6.2 Model Variations  (p.?)
    └─ [0021] 6.3 English Constituency Parsing 

In [20]:
# count total nodes
def count_nodes(nodes):
    total= len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total
total=count_nodes(pageindex_tree)
print(f"🔢 Total nodes in the document: {total}")

🔢 Total nodes in the document: 25


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [ ]:
#LLM Tree Search Function
def llm_tree_search(query: str, tree: list, model:str="qwen/qwen3.7-flash:free")-> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    #compress tree to save tikens only send titles + shirt summaries
    def compress(nodes):
        out=[]
        for n in nodes:
            entry={
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars of text
            }
            if n.get("nodes"):
                entry["children"]=compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree=compress(tree)
    # ... rest of the function
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

    Query: {query}

    Document Tree:
    {json.dumps(compressed_tree, indent=2)}

    Reply ONLY in this exact JSON format:
    {{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
    }}"""
    
    response=ai_client.chat.completions.create(model=model, messages=   [{"role": "user", "content": prompt}], response_format={"type": "json_object"})
    return json.loads(response.choices[0].message.content)#type: ignore

In [27]:

# ── Test with a sample query ─────────────────────────────────────────────────
query = "Why Self-Attention?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))


🔍 Query: Why Self-Attention?

🧠 LLM Reasoning:
The query asks 'Why Self-Attention?'. Looking at the document tree, there is a section explicitly titled '4 Why Self-Attention' with node_id '0016'. The summary for this node states: 'In this section we compare various aspects of self-attention layers to the recurrent and convolutional layers commonly used f...'. This directly addresses the reasoning behind using self-attention compared to other methods. While Section 3 (Model Architecture) describes how attention works, Section 4 specifically justifies *why* it is chosen. Therefore, node 0016 is the most relevant.

🎯 Selected Node IDs: ['0016']


---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [28]:

# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [35]:
def generate_answer(query: str, nodes: list, model: str = "qwen/qwen3.7-flash:free") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\\n\\n---\\n\\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = ai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000,          # ⬅️ Added this to prevent cutting off mid-sentence
        temperature=0.1           # ⬅️ Lower temperature keeps answers deterministic
    )
    
    return response.choices[0].message.content

In [36]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [37]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is the primary architectural component that the Transformer relies on, entirely dispensing with recurrence and convolutions?",
    tree=pageindex_tree
)

🔍 Query: What is the primary architectural component that the Transformer relies on, entirely dispensing with recurrence and convolutions?

🧠 Reasoning: The query asks for the primary architectural component of the Transformer that replaces recurrence and convolutions. 

1.  **Section 3.5 (Positional Encoding)** explicitly states: 'Since our model con...
🎯 Retrieved node IDs: ['0022']
📄 Sections found: ['7 Conclusion']

📝 Answer:
The primary architectural component is **multi-headed self-attention** (Section: '7 Conclusion', Page ?).


In [39]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "In the scaled dot-product attention formula, what is the purpose of dividing the dot products by $\\sqrt{d_k}$?",
    "Why does multi-head attention allow the model to jointly attend to information from different representation subspaces at different positions?",
    "Summarize the Attention Is All You Need paper's key contributions and findings.",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: In the scaled dot-product attention formula, what is the purpose of dividing the dot products by $\sqrt{d_k}$?
A: The purpose of dividing the dot products by $\sqrt{d_k}$ is to counteract the effect where large values of $d_k$ cause dot products to grow in magnitude, which pushes the softmax function into regions with extremely small gradients (Section: '3.2.1 Scaled Dot-Product Attention', Page ?)....
-------------------------------------------------------

Q: Why does multi-head attention allow the model to jointly attend to information from different representation subspaces at different positions?
A: Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions because it performs linear projections of queries, keys, and values $h$ times with different learned linear projections, then performs the attention function in paral...
-------------------------------------------------------

Q: Summarize the Attent

---
## 🎓 Section 6: Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.


In [40]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.

In [41]:

FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [42]:
# ── Expert Routing Rules — Attention Is All You Need Paper ───────────────────
# Vaswani et al. | 7 Core Sections | Structural RAG Index
ATTENTION_PAPER_RULES = """
Route queries to the correct section node using these rules:
 
NODE_001  Abstract & Introduction     → transformer motivation, eliminating recurrence, speed, BLEU scores
NODE_002  Background                  → ByteNet, ConvS2S, self-attention definition, comparison with CNNs/RNNs
NODE_003  3.1 Encoder-Decoder Stacks  → N=6 layers, d_model=512, sub-layers, residual connections, layer norm
NODE_004  3.2.1 Scaled Dot-Product    → Q, K, V matrices, formula, scaling factor 1/sqrt(d_k), softmax gradients
NODE_005  3.2.2 Multi-Head Attention  → h=8 heads, parallel projections, concatenation, W^O matrix dimensions
NODE_006  3.2.3 Attention Applications  → encoder-decoder attention, encoder self-attention, masked decoder self-attention
NODE_007  3.3 Feed-Forward Networks   → FFN(x) = max(0, xW_1 + b_1)W_2 + b_2, inner-layer d_ff=2048
NODE_008  3.4 Embeddings & Softmax    → learned embeddings, weight sharing, pre-softmax linear transformation
NODE_009  3.5 Positional Encoding     → sinusoidal functions, sine/cosine frequencies, relative/absolute position
NODE_010  4 Why Self-Attention        → computational complexity O(n^2 * d), sequential operations O(1), path lengths
NODE_011  5 Training                  → training regime, optimizer, dropout, hardware (P100 GPUs)
NODE_012  6.1 Machine Translation     → WMT 2014 English-to-German and English-to-French results, BLEU scores
NODE_013  6.2 Model Variations        → ablation studies, impact of changing heads d_k, d_v, dropout, positional embeddings
NODE_014  6.3 Constituency Parsing    → Penn Treebank WSJ dataset, generalization to other tasks
NODE_015  7 Conclusion                → summary of contributions, future work (images, audio, video modalities)
 
Cross-cutting rules:
- "formulas / math / scaled dot product"    → NODE_004 (Scaled Dot-Product) + NODE_005 (Multi-Head)
- "architecture / stack / dimensions"       → NODE_003 (Stacks) + NODE_007 (FFN) + NODE_009 (Positional Encoding)
- "why choose attention / complexity"       → NODE_002 (Background) + NODE_010 (Why Self-Attention)
- "results / benchmarks / BLEU"             → NODE_012 (Machine Translation) + NODE_014 (Parsing)
"""

In [43]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "qwen/qwen3.7-flash:free"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = ai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [45]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "How does the model handle the limitation where words appear sequentially and block parallel computing?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: How does the model handle the limitation where words appear sequentially and block parallel computing?

── Without Expert Rules ──
Nodes: ['0004', '0016', '0015']

── With Expert Rules ──
Nodes: ['0016', '0004', '0002']
Reasoning: The query asks about how the model handles the limitation of sequential computation (words appearing sequentially) which blocks parallel computing. This is a core architectural motivation for the Transformer model presented in 'Attention Is All You Need'. 

1. Node 0002 (Abstract) explicitly states 


In [48]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="Details of the syllabus of the Attention Is All You Need paper",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

The "Attention Is All You Need" paper proposes the Transformer architecture, which consists of an encoder and a decoder stack (Section 3 Model Architecture). The specific components and details are as follows:

**1. Encoder and Decoder Stacks (Section 3.1 Encoder and Decoder Stacks)**
*   **Structure:** Both encoder and decoder consist of a stack of $N = 6$ identical layers.
*   **Encoder Layers:** Each layer has two sub-layers:
    *   Multi-head self-attention mechanism.
    *   Position-wise fully connected feed-forward network.
*   **Decoder Layers:** Each layer has three sub-layers:
    *   Multi-head self-attention over the input sequence.
    *   Multi-head attention over the output of the encoder stack ("encoder-decoder attention").
    *   Position-wise fully connected feed-forward network.
*   **Connections:** Residual connections are used around each sub-layer, followed by Layer Normalization. The output formula for each sub-layer is $\text{LayerNorm}(x + \text{Sublayer}(x))

---
## 💬 Section 7: Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [49]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
I'll look at the document's structure first, then read the key sections.**Key findings of "Attention Is All You Need" (Vaswani et al., NIPS 2017)**

**1. Attention alone is sufficient**
The Transformer is the first sequence transduction model built *entirely* on attention — no recurrence, no convolutions. Encoder and decoder are 6-layer stacks of multi-head self-attention + position-wise feed-forward sublayers with residual connections and layer norm (p. 3).

**2. State-of-the-art translation at a fraction of the cost** (Table 2, p. 8)
- WMT 2014 **EN→DE: 28.4 BLEU** (big model) — beats all prior models *including ensembles* by >2.0 BLEU
- WMT 2014 **EN→FR: 41.8 BLEU** (41.0 reported in text), best single model, at <¼ the training cost of the prior SOTA
- The *base* model already surpasses all previously published models/ensembles
- Training: 12 hours (base) / 3.5 days (big) on 8 P100 GPUs; ~3.3·10¹⁸ FLOPs for base vs. 10²⁰–10²¹ for competitors

**3. Why self-attenti

In [51]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a multi-turn conversation about the Transformer architecture
questions = [
    # Turn 1: Core mechanism
    "How does scaled dot-product attention work, and why do we divide by the square root of d_k?",
    
    # Turn 2: Extension/Building on turn 1
    "How does multi-head attention build on that concept to look at multiple subspaces simultaneously?",
    
    # Turn 3: Implementation details
    "What specific hyperparameters and dimensions (like h and d_model) did they use for those attention heads?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: How does scaled dot-product attention work, and why do we divide by the square root of d_k?
🤖 Assistant: I'll look at the attention section of the paper.## Scaled Dot-Product Attention (§3.2.1, p. 4)

**The mechanism.** Attention maps a query plus a set of key–value pairs to an output that is a weighted sum of the values, where each value's weight comes from a compatibility function between the query and that value's key.

Inputs: queries and keys of dimension $d_k$, values of dimension $d_v$. Steps:...
-------------------------------------------------------

👤 User: How does multi-head attention build on that concept to look at multiple subspaces simultaneously?
🤖 Assistant: # Multi-Head Attention (§3.2.2, pp. 4–5)

**The core move.** Instead of running *one* attention function over full $d_{\text{model}}$-dimensional queries, keys, and values, the paper linearly projects them **$h$ times with different, learned linear projections** down to $d_k$, $d_k$, and $d_v$ dimensions.